# Quickstart Tutorial: End-to-End Spatial Analysis

Welcome to `spatial_pipeline`! This interactive tutorial demonstrates how to:
1. Ingest continuous molecule coordinates
2. Perform high-performance spatial binning
3. Build spatial graphs with GPU/CPU backends
4. Compute Spatially Variable Genes (Moran's I & Geary's C)
5. Export to AnnData and Zarr

In [ ]:
import spatial_pipeline as sp
import numpy as np
import pandas as pd

# Generate synthetic spatial transcripts
mock_csv = "demo_transcripts.csv"
sp.generate_vectorized_mock_data(mock_csv, num_molecules=25000)

# Load data with zero-copy PyArrow
df = sp.load_transcript_data(mock_csv, use_arrow=True)
print(f"Loaded {len(df):,} transcripts across {df['gene'].nunique()} genes.")
df.head()

## 2. Spatial Binning

Discretize continuous coordinates into 15µm square spatial bins.

In [ ]:
sparse_counts, bin_coords = sp.compute_square_bins(df, resolution=15.0, min_counts=1)
print(f"Active Bins: {sparse_counts.shape[0]:,}")
print(f"Genes:       {sparse_counts.shape[1]}")

## 3. GPU / CPU Spatial Graph Construction

Construct nearest-neighbor spatial adjacency and Graph Laplacian matrices.

In [ ]:
adj_matrix, laplacian = sp.build_knn_graph(bin_coords, n_neighbors=15, backend="auto")
print(f"Adjacency Non-Zero Edges: {adj_matrix.nnz:,}")

## 4. Spatially Variable Gene (SVG) Discovery

Rank genes by Moran's I spatial autocorrelation and calculate FDR p-values.

In [ ]:
gene_names = df['gene'].cat.categories.to_numpy()
svg_df = sp.compute_spatially_variable_genes(sparse_counts, gene_names, adj_matrix)
svg_df

## 5. AnnData & Zarr Interoperability

In [ ]:
# Convert to AnnData
adata = sp.to_anndata(
    sparse_counts=sparse_counts,
    bin_coords=bin_coords,
    genes=gene_names,
    adjacency_matrix=adj_matrix,
    laplacian_matrix=laplacian,
    compute_svg=True
)
print(adata)

# Export to chunked Zarr hierarchy
sp.to_zarr(adata, "spatial_output.zarr")
print("Exported to spatial_output.zarr successfully!")